# Balanced DuoDiT Sample Generation

Generate an exactly class-balanced image batch for global and per-class evaluation.

This notebook writes all PNGs into one flat run folder. Class membership remains available through each filename, `samples.csv`, `metadata.json`, and `arr_1` in the generated NPZ. The NPZ keeps ADM compatibility through its `arr_0` image array.

Prerequisites:
- A CUDA runtime and a DuoDiT checkpoint containing full EMA weights.
- Enough disk for all PNGs, temporary NPZ construction, and the final NPZ.
- Run from the DuoDiT repository root, or use the hosted-runtime bootstrap below.

## Workflow

1. Detect local, Colab, or Kaggle execution and prepare the repository.
2. Configure the checkpoint, classes, sample count, sampler, and hardware.
3. Validate exact class balance and preview the `torchrun` command.
4. Opt in to generation.
5. Validate the flat PNG folder, manifest, metadata, and labeled NPZ.
6. Preview a small image grid.

This notebook intentionally does not compute FID or other metrics.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shlex
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
HOSTED_RUNTIME = IN_COLAB or IN_KAGGLE
RUNTIME_NAME = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"
REPO_URL = "https://github.com/mrdjango/DuoDiT.git"

print(f"Runtime: {RUNTIME_NAME}")

if not Path("sample_balanced_ddp.py").exists():
    if not HOSTED_RUNTIME:
        raise FileNotFoundError(
            "sample_balanced_ddp.py was not found. Run this notebook from the DuoDiT repository root."
        )
    repo_root = Path("/content/DuoDiT") if IN_COLAB else Path("/kaggle/working/DuoDiT")
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    os.chdir(repo_root)

REPO_ROOT = Path.cwd().resolve()
required_files = ["sample_balanced_ddp.py", "models.py", "download.py", "diffusion"]
missing_files = [name for name in required_files if not (REPO_ROOT / name).exists()]
if missing_files:
    raise FileNotFoundError(f"DuoDiT repository is incomplete. Missing: {missing_files}")

package_modules = {
    "diffusers": "diffusers",
    "timm": "timm",
    "PIL": "pillow",
    "tqdm": "tqdm",
}
missing_packages = [package for module, package in package_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages and HOSTED_RUNTIME:
    # Keep the hosted runtime's CUDA-compatible torch and torchvision builds.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
elif missing_packages:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing_packages) +
        ". Install them in the active environment without replacing its CUDA PyTorch build."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")


## Configure Generation

`NUM_SAMPLES` must be divisible by the number of selected classes. With all 1,000 ImageNet classes and 50,000 samples, the generator creates exactly 50 images per class.

Set `CLASS_IDS = None` for all classes or provide an explicit list such as `[972, 973, 974, 975, 976]`.

Use an empty, new output location for every run. The sampler rejects non-empty run folders and existing sibling NPZ files.

In [ ]:
import torch

# Model and checkpoint
MODEL = "DiT-XL/2"
CKPT = ""  # Required: path to a DuoDiT checkpoint with full EMA weights.
IMAGE_SIZE = 256
NUM_CLASSES = 1000

# None means all class IDs from 0 to NUM_CLASSES - 1.
# Example subset: CLASS_IDS = [972, 973, 974, 975, 976]
CLASS_IDS = None
NUM_SAMPLES = 50_000

# Standard DiT FID generation defaults.
NUM_SAMPLING_STEPS = 250
VAE = "ema" #mse
CFG_SCALE = 1.0
GLOBAL_SEED = 0
TF32 = True

# Hardware and output.
NUM_GPUS = torch.cuda.device_count()
PER_PROC_BATCH_SIZE = 16
SAMPLE_ROOT = "balanced_duodit_samples"

# Safety gate. Keep False until the validation and command preview look correct.
RUN_GENERATION = False


## Validate Configuration and Balance

This validation runs before `torchrun`, model construction, or checkpoint loading. A non-divisible request fails here rather than silently assigning extra samples to some classes.

In [ ]:
from sample_balanced_ddp import (
    build_run_name,
    build_sample_records,
    normalize_classes,
    validate_balanced_request,
)

if not CKPT:
    raise ValueError("Set CKPT to a DuoDiT checkpoint before generating samples.")
checkpoint_path = Path(CKPT).expanduser()
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Checkpoint does not exist: {checkpoint_path}")
if IMAGE_SIZE not in {256, 512}:
    raise ValueError("IMAGE_SIZE must be 256 or 512.")
if VAE not in {"mse", "ema"}:
    raise ValueError("VAE must be 'mse' or 'ema'.")
if CFG_SCALE < 1.0:
    raise ValueError("CFG_SCALE must be at least 1.0.")
if NUM_GPUS < 1 or not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required for DDP sampling.")
if PER_PROC_BATCH_SIZE < 1:
    raise ValueError("PER_PROC_BATCH_SIZE must be at least 1.")

SELECTED_CLASSES = normalize_classes(CLASS_IDS, NUM_CLASSES)
SAMPLES_PER_CLASS = validate_balanced_request(NUM_SAMPLES, SELECTED_CLASSES)
EXPECTED_RECORDS = build_sample_records(NUM_SAMPLES, SELECTED_CLASSES)
EXPECTED_COUNTS = Counter(record.class_id for record in EXPECTED_RECORDS)

print(f"Selected classes : {len(SELECTED_CLASSES):,}")
print(f"Total samples    : {NUM_SAMPLES:,}")
print(f"Samples/class    : {SAMPLES_PER_CLASS:,}")
print(f"GPUs             : {NUM_GPUS}")
print(f"Per-GPU batch    : {PER_PROC_BATCH_SIZE}")
print(f"First labels     : {[record.class_id for record in EXPECTED_RECORDS[:12]]}")
print(f"Equal counts     : {set(EXPECTED_COUNTS.values()) == {SAMPLES_PER_CLASS}}")


## Build and Inspect the Command

The generated run directory name records the checkpoint, image size, VAE, steps, CFG, classes, sample count, and seed. All generated PNGs are written directly inside that one directory.

In [ ]:
from argparse import Namespace

run_args = Namespace(
    model=MODEL,
    ckpt=str(checkpoint_path),
    image_size=IMAGE_SIZE,
    vae=VAE,
    num_sampling_steps=NUM_SAMPLING_STEPS,
    cfg_scale=CFG_SCALE,
    num_classes=NUM_CLASSES,
    num_samples=NUM_SAMPLES,
    global_seed=GLOBAL_SEED,
)
RUN_NAME = build_run_name(run_args, SELECTED_CLASSES)
RUN_DIR = REPO_ROOT / SAMPLE_ROOT / RUN_NAME
NPZ_PATH = Path(f"{RUN_DIR}.npz")

command = [
    "torchrun",
    "--standalone",
    "--nnodes=1",
    f"--nproc_per_node={NUM_GPUS}",
    "sample_balanced_ddp.py",
    "--model", MODEL,
    "--ckpt", str(checkpoint_path),
    "--image-size", str(IMAGE_SIZE),
    "--num-classes", str(NUM_CLASSES),
    "--num-samples", str(NUM_SAMPLES),
    "--num-sampling-steps", str(NUM_SAMPLING_STEPS),
    "--vae", VAE,
    "--cfg-scale", str(CFG_SCALE),
    "--global-seed", str(GLOBAL_SEED),
    "--per-proc-batch-size", str(PER_PROC_BATCH_SIZE),
    "--sample-dir", SAMPLE_ROOT,
]
if not TF32:
    command.append("--no-tf32")
if CLASS_IDS is not None:
    command.append("--classes")
    command.extend(str(class_id) for class_id in SELECTED_CLASSES)

print("Command:")
print(shlex.join(command))
print("\nExpected artifacts:")
print(f"PNG folder : {RUN_DIR}")
print(f"Manifest   : {RUN_DIR / 'samples.csv'}")
print(f"Metadata   : {RUN_DIR / 'metadata.json'}")
print(f"NPZ        : {NPZ_PATH}")
print(f"First PNG  : {RUN_DIR / EXPECTED_RECORDS[0].filename}")


## Generate Samples

Set `RUN_GENERATION = True` in the configuration cell only after reviewing the command and expected balance. Model construction uses the repository's existing `models.py`, including its current diagnostic output.

In [ ]:
if RUN_GENERATION:
    print(shlex.join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Generation skipped. Set RUN_GENERATION = True after validating the configuration.")


## Validate Generated Artifacts

The checks below do not load the large image array into RAM. They verify exact filenames, equal class counts, manifest ordering, metadata, NPZ headers, and the smaller NPZ label array.

In [ ]:
import csv
import numpy as np

from sample_balanced_ddp import validate_generated_samples


def inspect_npz_headers(path: Path) -> dict[str, dict[str, object]]:
    arrays = {}
    with zipfile.ZipFile(path) as archive:
        for member in archive.namelist():
            if not member.endswith(".npy"):
                continue
            with archive.open(member) as stream:
                version = np.lib.format.read_magic(stream)
                if version == (1, 0):
                    shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(stream)
                else:
                    shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(stream)
            arrays[Path(member).stem] = {
                "shape": tuple(shape),
                "dtype": str(dtype),
                "fortran_order": bool(fortran_order),
            }
    return arrays


if not RUN_DIR.exists():
    raise FileNotFoundError(f"Generated run directory does not exist: {RUN_DIR}")
if not NPZ_PATH.exists():
    raise FileNotFoundError(f"Generated NPZ does not exist: {NPZ_PATH}")

class_counts = validate_generated_samples(RUN_DIR, EXPECTED_RECORDS)
if set(class_counts.values()) != {SAMPLES_PER_CLASS}:
    raise ValueError(f"Unexpected class counts: {class_counts}")

manifest_path = RUN_DIR / "samples.csv"
with manifest_path.open(newline="", encoding="utf-8") as manifest_file:
    manifest_rows = list(csv.DictReader(manifest_file))
if len(manifest_rows) != NUM_SAMPLES:
    raise ValueError(f"Manifest has {len(manifest_rows)} rows, expected {NUM_SAMPLES}")
for record, row in zip(EXPECTED_RECORDS, manifest_rows):
    if (
        int(row["index"]) != record.index
        or row["filename"] != record.filename
        or int(row["class_id"]) != record.class_id
    ):
        raise ValueError(f"Manifest mismatch at global index {record.index}")

metadata = json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))
if metadata["status"] != "complete" or metadata["num_samples"] != NUM_SAMPLES:
    raise ValueError("Metadata does not describe a complete expected run")

headers = inspect_npz_headers(NPZ_PATH)
expected_image_shape = (NUM_SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 3)
if headers.get("arr_0", {}).get("shape") != expected_image_shape:
    raise ValueError(f"Unexpected arr_0 shape: {headers.get('arr_0')}")
if headers.get("arr_0", {}).get("dtype") != "uint8":
    raise ValueError(f"Unexpected arr_0 dtype: {headers.get('arr_0')}")
if headers.get("arr_1", {}).get("shape") != (NUM_SAMPLES,):
    raise ValueError(f"Unexpected arr_1 shape: {headers.get('arr_1')}")

with np.load(NPZ_PATH) as archive:
    labels = archive["arr_1"]
label_counts = Counter(int(label) for label in labels)
if label_counts != Counter(EXPECTED_COUNTS):
    raise ValueError("NPZ arr_1 labels do not match the expected balanced distribution")

print("Validation passed.")
print(f"PNGs            : {NUM_SAMPLES:,}")
print(f"Classes         : {len(class_counts):,}")
print(f"Samples/class   : {SAMPLES_PER_CLASS:,}")
print(f"NPZ arr_0       : {headers['arr_0']}")
print(f"NPZ arr_1       : {headers['arr_1']}")


## Preview Samples

This final cell displays a small grid selected from the flat run directory. Filenames include the class ID, so individual class samples can be located later without creating permanent class folders.

In [ ]:
from IPython.display import display
from PIL import Image as PILImage, ImageDraw

PREVIEW_COUNT = 12
preview_records = EXPECTED_RECORDS[: min(PREVIEW_COUNT, len(EXPECTED_RECORDS))]
tiles = []
for record in preview_records:
    with PILImage.open(RUN_DIR / record.filename) as image:
        tile = image.convert("RGB").copy()
    draw = ImageDraw.Draw(tile)
    draw.rectangle((0, 0, 110, 20), fill="black")
    draw.text((4, 4), f"class {record.class_id}", fill="white")
    tiles.append(tile)

columns = min(4, len(tiles))
rows = math.ceil(len(tiles) / columns)
width, height = tiles[0].size
grid = PILImage.new("RGB", (columns * width, rows * height), "white")
for position, tile in enumerate(tiles):
    grid.paste(tile, ((position % columns) * width, (position // columns) * height))

display(grid)


## Output Contract

- One flat configuration-named run directory.
- PNG names: `000000-class0000.png`.
- `samples.csv`: global index, filename, and class ID.
- `metadata.json`: sampler settings and verified class histogram.
- Sibling `.npz`: `arr_0` contains ordered uint8 RGB images; `arr_1` contains aligned int64 class IDs.

The existing evaluation notebooks are intentionally untouched.